# Duhamel Transport Beta Guide 01: Model Construction and 2D Signals

This notebook is the entry point for collaborators. It walks through the four-level exciton dimer used in the beta release and shows how the reference and target spectra are generated before any transport approximation is applied.

The important convention is that `lambda` is a dimensionless displacement from a reference, best-guess model:

\[
p_\mu(\lambda_\mu) = p_{\mu,\mathrm{ref}} + \delta_\mu \lambda_\mu.
\]

The controls selected in this demo are `J_12` and `k_12`. Other Hamiltonian and dissipative parameters are held fixed at their reference values.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys

import numpy as np
import matplotlib.pyplot as plt

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import Liouville_transport_general as general
from demo_dimer_duhamel_transport import dimer_parameter_map_from_args, normalize_to_unit_power
from exciton_dimer_model import exciton_dimer_model, one_exciton_summary, detailed_balance_site_transfer_rates
from fit_v_system_transport_signal import pathway_signal_grid


## 1. Define the reference model

The beta demo uses a clean reference dimer: no coherent coupling, no incoherent inter-site transfer, equal site dipoles, equal linewidths, and ESA enabled. This makes the reference spectrum easy to interpret: two diagonal features from the two site excitations, with no coherent cross-peak structure at `J_12 = 0`.


In [ ]:
args = SimpleNamespace(
    clean_reference_dimer=True,
    clean_gamma=0.01,
    clean_gamma_phi=0.01,
    clean_mu_esa_scale=1.0,
    delta_J=0.045,
    delta_k=0.012,
)

model = exciton_dimer_model()
parameter_map = dimer_parameter_map_from_args(args)

reference_lambda = parameter_map.zero()
target_lambda = np.array([-0.25, 0.25])

reference_params = parameter_map.params(reference_lambda)
target_params = parameter_map.params(target_lambda)

print("controls:")
for control in parameter_map.controls:
    print(f"  {control.name}: reference={control.reference_value:g}, delta={control.scale:g}")

print("\ntarget lambda:", target_lambda)
print("physical displacements:", parameter_map.physical_displacements(target_lambda))
print("target J_12, k_12:", target_params["J_12"], target_params["k_12"])


## 2. Inspect the one-exciton block

This is a quick sanity check on the physical model. Changing `J_12` mixes the site excitations and redistributes oscillator strength. In this site-basis implementation the transition dipole couples the common ground state to both `|10>` and `|01>`.


In [ ]:
for name, params in [("reference", reference_params), ("target", target_params)]:
    print(f"\n{name}")
    print("detailed-balance transfer rates k_1->2, k_2->1:", detailed_balance_site_transfer_rates(params))
    for item in one_exciton_summary(params):
        print(item)


## 3. Build phase-matched pathway bases

The code labels pathways by phase-matching sector and by the left/right dipole action sequence. RP and NRP sectors are kept separate. This implements wave-vector conservation at the transport-matrix level rather than allowing a global least-squares projection to mix sectors.


In [ ]:
omega = np.linspace(0.8, 1.4, 32)
tau2 = 1.0
eta = 3e-3
sectors = ("RP", "NRP")

sector_pathways = {}
for sector in sectors:
    candidates = general.canonical_pathways((sector,))
    pathways, residuals = general.independent_canonical_pathways(
        model,
        parameter_map,
        omega,
        candidates,
        stage="after_G3",
        tau2=tau2,
        eta=eta,
        rtol=1e-9,
    )
    sector_pathways[sector] = pathways
    print(f"{sector}: retained {len(pathways)} of {len(candidates)} candidate pathways")
    for idx, pathway in enumerate(pathways):
        print(f"  {idx}: {pathway.label}")


## 4. Compute reference and target spectra

The spectra below are direct model calculations. They are the “model experimental spectra” used to test the transport approximation later.


In [ ]:
def sector_spectrum(lam, sector):
    pathway_signals = pathway_signal_grid(
        model,
        parameter_map,
        lam,
        omega,
        omega,
        sector_pathways[sector],
        tau2=tau2,
        eta=eta,
    )
    return np.sum(pathway_signals, axis=1)

reference_signal = sum(sector_spectrum(reference_lambda, sector) for sector in sectors)
target_signal = sum(sector_spectrum(target_lambda, sector) for sector in sectors)

reference_signal, reference_power = normalize_to_unit_power(reference_signal, omega, omega)
target_signal, target_power = normalize_to_unit_power(target_signal, omega, omega)

print(f"reference integrated |S|^2 before normalization: {reference_power:.6e}")
print(f"target integrated |S|^2 before normalization:    {target_power:.6e}")


In [ ]:
def contour_signal(ax, signal, title, component="real"):
    grid = signal.reshape(len(omega), len(omega))
    if component == "real":
        values = np.real(grid)
        vmax = max(abs(values.min()), abs(values.max()))
        levels = np.linspace(-vmax, vmax, 21)
        image = ax.contourf(omega, omega, values, levels=levels, cmap="RdBu_r")
    elif component == "imag":
        values = np.imag(grid)
        vmax = max(abs(values.min()), abs(values.max()))
        levels = np.linspace(-vmax, vmax, 21)
        image = ax.contourf(omega, omega, values, levels=levels, cmap="RdBu_r")
    else:
        values = np.abs(grid)
        levels = np.linspace(0.0, values.max(), 21)
        image = ax.contourf(omega, omega, values, levels=levels, cmap="magma")
    ax.set_title(title)
    ax.set_xlabel(r"$\omega_1$ / eV")
    ax.set_ylabel(r"$\omega_3$ / eV")
    return image

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for row, (name, signal) in enumerate([("reference", reference_signal), ("target", target_signal)]):
    for col, component in enumerate(["real", "imag", "abs"]):
        image = contour_signal(axes[row, col], signal, f"{name}: {component}", component)
        fig.colorbar(image, ax=axes[row, col], shrink=0.85)
plt.show()


## 5. Takeaway

At this stage nothing has been approximated by Duhamel transport. We have only specified the model, selected a reference point and control displacements, retained independent phase-matched pathways, and computed exact model spectra. The next notebook uses these same objects to construct and analyze the transport matrix.
